## CS5242 Neural Networks and Deep Learning
## Sem 2 2025/26
### Lecturer: Xavier Bresson
### Teaching Assistants: Yuwei Zeng, KinWhye Chew, Joel Loo Enming, Ryoji Kubo

## Midterm exam, coding test (Advanced Application)
Date: Mar 2 2026 <br>
Time: 6:30pm-8:00pm (90min) <br>

*Instructions* <br>
Name: Please, add your name here : <br>
Answers: Please write your answers directly in this notebook by completing the code sections marked with  
`# YOUR CODE STARTS HERE`  
`# YOUR CODE` (it can span one or multiple lines)  
`# YOUR CODE ENDS HERE`. <br>
Remark: Ensure your code runs without errors. No points will be awarded for buggy or incomplete code.  
Remark: If certain conditions of the questions (for eg. hyperparameter values) are not stated, you are free to choose anything you want.

## Exercise 1 : Implement Layer Normalization

In this exercise, you will manually implement **Layer Normalization**, an essential normalization technique used in Transformers that helps stabilize the training of deep networks.

Given an input tensor $X \in \mathbb{R}^{B \times N \times D}$ where $B$ is batch size, $N$ is sequence length, and $D$ is embedding dimension, Layer Normalization normalizes the values across the last dimension $D$:
$$
\mu = \frac{1}{D} \sum_{i=1}^{D} X_i, \quad \sigma^2 = \frac{1}{D} \sum_{i=1}^{D} (X_i - \mu)^2
$$
$$
X_{\text{norm}} = \frac{X - \mu}{\sqrt{\sigma^2 + \epsilon}}
$$
$$
Y = \gamma X_{\text{norm}} + \beta
$$

**[Your Tasks]**

Please complete the `layer_norm_manual()` function:
1) Calculate the mean and variance across the last dimension (`dim=-1`). You must ensure dimensions are kept for broadcasting using `keepdim=True` and variance computation is biased (`unbiased=False`).
2) Normalize the tensor using the variance and $\epsilon$.
3) Apply the learnable scale $\gamma$ and shift $\beta$.

**[Implementation Criteria]**

The evaluation checks if your manually normalized tensor matches the expected transformation. If correct, the script prints **"Well Done!"**.

**[Useful Functions]**

- `tensor.mean(dim, keepdim)` Returns the mean value.
- `tensor.var(dim, keepdim, unbiased)` Returns the variance. 
- `torch.sqrt(tensor)` Computes the square root element-wise.

In [8]:
%reset -f
import datetime
import torch
print('Timestamp:',datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S"))

def layer_norm_manual(x, gamma, beta, eps=1e-5):
    
    #############################################
    # YOUR CODE STARTS HERE
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    x_norm = (x - mean) / torch.sqrt(var + eps)
    out = gamma * x_norm + beta
    # YOUR CODE ENDS HERE
    #############################################
    
    return out

torch.manual_seed(42)
x = torch.randn(2, 3, 4)
gamma = torch.ones(4) * 1.5
beta = torch.zeros(4) - 0.5
out = layer_norm_manual(x, gamma, beta)

expected_sum = torch.tensor(-6.0)

if out.shape == (2, 3, 4) and (out.sum() - expected_sum).abs() < 1e-3:
    print("Well Done!")
else:
    print("Try again.")

Timestamp: 26-04-12--14-29-37
Try again.


## Exercise 2 : Causal Masked Self-Attention

In autoregressive generation tasks like Language Modeling, **Causal Masking** is applied to the Self-Attention mechanism to prevent tokens from attending to future tokens.

The causal attention equation introduces a mask matrix $M$:
$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}} + M\right) V
$$
where $M_{i, j} = 0$ if $i \geq j$ (allowed to attend to past and present), and $M_{i, j} = -\infty$ if $i < j$ (future tokens are masked). Note that applying softmax to $-\infty$ yields $0$.

**[Your Tasks]**

Complete the `causal_attention()` function:
1) Compute the raw scaled attention scores $S = \frac{Q K^T}{\sqrt{d_k}}$.
2) Create a lower-triangular causal mask.
3) Use `masked_fill` to replace values in $S$ corresponding to future tokens (where mask is 0) with `float('-inf')`.
4) Apply softmax and matrix multiplication with $V$.

**[Implementation Criteria]**

The test checks your attention weights matrix. The upper triangular elements must be absolute zeros, matching the expected output.

**[Useful Functions]**

- `torch.tril(tensor)` Returns the lower triangular part of the matrix.
- `tensor.masked_fill(mask, value)` Fills elements of `tensor` with `value` where `mask` is True.

In [ ]:
%reset -f
import datetime
import torch
import math
print('Timestamp:',datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S"))

def causal_attention(Q, K, V):
    seq_len = Q.size(0)
    d_k = Q.size(-1)
    
    #############################################
    # YOUR CODE STARTS HERE
    attn_scores = (Q @ K.transpose(-2,-1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    mask = torch.tril(torch.ones(seq_len,seq_len))
    masked_filled = attn_scores.masked_fill(mask==0, -1e9 )
    attn_weights = torch.softmax(masked_filled  , dim=-1)
    out = attn_weights @ V

    # YOUR CODE ENDS HERE
    #############################################
    
    return out, attn_weights

torch.manual_seed(42)
Q = torch.randn(4, 8)
K = torch.randn(4, 8)
V = torch.randn(4, 8)

out, attn = causal_attention(Q, K, V)

expected_attn = torch.tensor([
    [1.0000, 0.0000, 0.0000, 0.0000],
    [0.0585, 0.9415, 0.0000, 0.0000],
    [0.2114, 0.4180, 0.3706, 0.0000],
    [0.1926, 0.1803, 0.1955, 0.4316]
])

if attn.shape == (4, 4) and out.shape == (4, 8) and (attn - expected_attn).abs().sum() < 1e-2:
    print("Well Done!")
else:
    print("Try again.")

Timestamp: 26-04-12--14-48-05
Well Done!


## Exercise 3 : Channel Attention (Squeeze-and-Excitation)

In advanced CNN architectures, **Channel Attention** mechanisms (such as Squeeze-and-Excitation networks) help the model dynamically re-calibrate channel-wise feature dependencies.

Given a feature map $X \in \mathbb{R}^{B \times C \times H \times W}$:
1. **Squeeze**: Perform a Global Average Pooling operation across spatial dimensions $(H, W)$ to produce a channel descriptor $Z \in \mathbb{R}^{B \times C}$.
2. **Excitation**: Pass $Z$ through a two-layer bottleneck MLP ($W_1 \in \mathbb{R}^{\frac{C}{r} \times C}$, $W_2 \in \mathbb{R}^{C \times \frac{C}{r}}$ with a ReLU activation in between, and a Sigmoid activation at the end) to obtain gating weights $S \in \mathbb{R}^{B \times C}$.
3. **Scale**: Broadcast $S$ back to the spatial dimensions and scale the original input $Y = S \cdot X$.

**[Your Tasks]**

In `forward()`, implement the Squeeze, Excitation, and Scaling steps:
1) Calculate Spatial Global Average Pooling on `x`.
2) Propagate through `self.fc1`, `self.relu`, `self.fc2`, and `self.sigmoid` to obtain the gating weights.
3) Un-squeeze and reshape the gating tensor so it can be multiplied element-wise with the original input feature map `x`.

**[Implementation Criteria]**

Runs the Squeeze-and-Excitation bottleneck on an input image tensor and validates the scaling values against our precalculated tensor sum.

**[Useful Functions]**

- `tensor.mean(dim)`
- `tensor.view(*shape)`

In [11]:
%reset -f
import datetime
import torch
import torch.nn as nn
print('Timestamp:',datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S"))

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=2):
        super().__init__()
        self.fc1 = nn.Linear(in_channels, in_channels // reduction, bias=False) # 4 -> 2
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(in_channels // reduction, in_channels, bias=False) # 2 -> 1
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        B, C, H, W = x.shape
        
        #############################################
        # YOUR CODE STARTS HERE
        # spatial Global Average Pooling
        z = x.mean(dim = (2,3)) # Across H, W -> last 2 dimensions 
        z = self.fc1(z)
        z = self.relu(z)
        z = self.fc2(z)
        S = self.sigmoid(z)
        out = x * S.view(B,C,1,1)
        
        # YOUR CODE ENDS HERE
        #############################################
        
        return out

torch.manual_seed(42)
ca = ChannelAttention(in_channels=4, reduction=2)
with torch.no_grad():
    ca.fc1.weight.copy_(torch.randn_like(ca.fc1.weight) * 0.1)
    ca.fc2.weight.copy_(torch.randn_like(ca.fc2.weight) * 0.1)
    
x = torch.randn(2, 4, 3, 3)
output = ca(x)
expected_sum = torch.tensor(1.6399)

if output.shape == (2, 4, 3, 3) and (output.sum() - expected_sum).abs() < 1e-3:
    print("Well Done!")
else:
    print("Try again.")

Timestamp: 26-04-12--15-24-28
Well Done!


## Exercise 4 : Vision Transformer (ViT) Sequence Preparation

To bridge Computer Vision and Transformers, **Vision Transformers (ViT)** process an image by splitting it into raw pixel patches. These patches are flattened, linearly mapped, and treated exactly like tokens in a sentence.

Given an input image $X \in \mathbb{R}^{B \times C \times H \times W}$ and a square patch size $P$:
1. Divide the image into $N = (\frac{H}{P} \times \frac{W}{P})$ non-overlapping patches. Each patch contains $C \times P \times P$ pixels.
2. Flatten the patches spatially into a sequence matrix of size $B \times N \times (C \cdot P^2)$ and apply a linear projection `self.proj` to map size to $d_{\text{model}}$.
3. Prepend a learnable classification token `[CLS]` (expanded across the batch size) at the beginning of the sequence, making the sequence length $N+1$.
4. Add the learnable absolute positional embeddings `self.pos_emb` (of size $1 \times (N+1) \times d_{\text{model}}$) to everyone.

**[Your Tasks]**

In the `forward()` method of `ViTPrep`:
1) Extract patches from `x` and format them to `(B, num_patches, patch_dim)`. You may use sequences of `view()` and `permute()`, combined with `contiguous()`.
2) Project the patches linearly using `self.proj`.
3) Apply `expand` on `self.cls_token` to make the batch dimension match, then concatenate it with the sequence.
4) Add `self.pos_emb`.

**[Implementation Criteria]**

Runs correct image patching and sequence construction. Validates final output tensor shape `(B, num_patches+1, d_model)` and numerical value.

**[Useful Functions]**

- `tensor.view(*shape)`
- `tensor.permute(*dims)`
- `torch.cat([tensor1, tensor2], dim)`
- `tensor.expand(size)`

In [ ]:
%reset -f
import datetime
import torch
import torch.nn as nn
print('Timestamp:',datetime.datetime.now().strftime("%y-%m-%d--%H-%M-%S"))

class ViTPrep(nn.Module):
    def __init__(self, image_size, patch_size, in_channels, d_model):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        patch_dim = in_channels * patch_size * patch_size
        
        self.proj = nn.Linear(patch_dim, d_model)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_emb = nn.Parameter(torch.randn(1, self.num_patches + 1, d_model))
        
    def forward(self, x):
        B, C, H, W = x.shape
        P = self.patch_size
        
        #############################################
        # YOUR CODE STARTS HERE

        x.
        
        # YOUR CODE ENDS HERE
        #############################################
        
        return out

torch.manual_seed(42)
vit = ViTPrep(image_size=8, patch_size=4, in_channels=3, d_model=16)

with torch.no_grad():
    vit.proj.weight.copy_(torch.ones_like(vit.proj.weight) * 0.05)
    vit.proj.bias.copy_(torch.zeros_like(vit.proj.bias))

x = torch.arange(2*3*8*8, dtype=torch.float32).view(2, 3, 8, 8) / 100.0
output = vit(x)

expected_sum = torch.tensor(579.5348)

if output.shape == (2, 5, 16) and (output.sum() - expected_sum).abs() < 1e-3:
    print("Well Done!")
else:
    print("Try again.")

## End of coding test